In [49]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

In [50]:
df = pd.read_excel(r"C:\Users\yapen\Desktop\TrainDataset2025.xls")

In [51]:
df = df.drop(columns=["ID"])
df = df.drop(columns=['pCR (outcome)'])

In [52]:
df = df.replace(999, np.nan)

In [53]:
df["HistologyType"] = df["HistologyType"].map({1:0, 2:1})
ordinal_cols = ["TumourStage", "Proliferation", "ChemoGrade"]
df[ordinal_cols] = df[ordinal_cols].astype("Int64")

In [54]:
mask = df.isna().sum(axis=1) > 1
df = df.drop(df[mask].index)

In [55]:
df = df.dropna(subset=['LNStatus'])

In [56]:
X = df.drop(columns=['RelapseFreeSurvival (outcome)'])

y = df['RelapseFreeSurvival (outcome)']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

In [57]:
gene_col = "Gene"
# -------- TRAIN SET --------
mask_train_known   = X_train[gene_col].notna() # true where gene present
mask_train_missing = X_train[gene_col].isna() # true where gene is NaN

# numeric predictors excluding gene
feat_cols = X_train.select_dtypes(include="number").columns.drop(gene_col)

X_known = X_train.loc[mask_train_known, feat_cols]
y_known = X_train.loc[mask_train_known, gene_col]
X_missing_train = X_train.loc[mask_train_missing, feat_cols]

# Train classifier on TRAIN ONLY
clf = RandomForestClassifier(random_state=0)
clf.fit(X_known, y_known)

# Fill missing Gene IN TRAIN
pred_gene_train = clf.predict(X_missing_train)
X_train.loc[mask_train_missing, gene_col] = pred_gene_train


# -------- TEST SET: apply classifier (NO FITTING) --------
mask_test_missing = X_test[gene_col].isna()
X_missing_test = X_test.loc[mask_test_missing, feat_cols]

if mask_test_missing.sum() > 0:
    pred_gene_test = clf.predict(X_missing_test)
    X_test.loc[mask_test_missing, gene_col] = pred_gene_test

In [58]:
X_train_df = pd.DataFrame(X_train, columns=X_train.columns)
X_test_df  = pd.DataFrame(X_test,  columns=X_test.columns)


Q1 = X_train_df.quantile(0.25)
Q3 = X_train_df.quantile(0.75)
IQR = Q3 - Q1

X_train_clip = X_train_df.clip(
    lower=Q1 - 1.5 * IQR,
    upper=Q3 + 1.5 * IQR,
    axis=1
)

X_test_clip = X_test_df.clip(
    lower=Q1 - 1.5 * IQR,
    upper=Q3 + 1.5 * IQR,
    axis=1
)

In [59]:
rf = RandomForestRegressor(
    n_estimators=38,
    max_depth=2,
    min_samples_split=5,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    max_samples=None,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train_clip, y_train)

y_pred_train = rf.predict(X_train_clip)
y_pred_test  = rf.predict(X_test_clip)

print("Train MAE (orig): {:.4f}".format(
    mean_absolute_error(y_train, y_pred_train)
))
print("Test  MAE (orig): {:.4f}".format(
    mean_absolute_error(y_test,  y_pred_test)
))

Train MAE (orig): 19.1895
Test  MAE (orig): 21.8409


In [60]:
# Train MAE (orig): 19.1895
# Test  MAE (orig): 21.8409